In [16]:
# !pip install tensorflow
# !pip install tensorflow_datasets

In [17]:
import tensorflow as tf
import tensorflow_datasets as tfds
# import seaborn as sns
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import itertools

print(tf.__version__)

2.19.0


In [18]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [19]:
# Shuffle the training data first
indices = tf.random.shuffle(tf.range(len(x_train)))
x_train_shuffled = tf.gather(x_train, indices)
y_train_shuffled = tf.gather(y_train, indices)

# Split into training and validation (80/20)
val_split = 0.2
num_train = int(len(x_train_shuffled) * (1 - val_split))
x_train_final = x_train_shuffled[:num_train]
y_train_final = y_train_shuffled[:num_train]
x_val = x_train_shuffled[num_train:]
y_val = y_train_shuffled[num_train:]

print(f"Training set size: {len(x_train_final)}")
print(f"Validation set size: {len(x_val)}")
print(f"Test set size: {len(x_test)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [20]:
dataset, info = tfds.load("cifar10", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

In [21]:
sample_images = (x_train[:4].astype('float32')) / 255.0  # Normalize to 0-1
# Resize to 299x299 for Xception
images_resized = tf.image.resize(sample_images, (299, 299))

In [22]:
inputs = tf.keras.applications.xception.preprocess_input(images_resized)

In [23]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
tf.keras.layers.RandomRotation(factor=0.05, seed=42),
tf.keras.layers.RandomContrast(factor=0.2, seed=42),
tf.keras.layers.RandomZoom(0.1, seed=42),
tf.keras.layers.RandomTranslation(0.1, 0.1, seed=42)  # add this
])


In [24]:
batch_size = 32
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])

train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)).map(lambda X, y: (preprocess(data_augmentation(X, training=True)), y)).batch(batch_size)
valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
test_set = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)

In [25]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
avg = tf.keras.layers.Dense(256, activation="relu")(avg)
avg = tf.keras.layers.Dropout(0.2)(avg)
output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)
Model = tf.keras.Model(inputs=base_model.input, outputs=output)
for layer in base_model.layers:
    layer.trainable = False

In [26]:
print(len(base_model.layers))

132


In [31]:
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = Model.fit(train_set, validation_data=valid_set, epochs=5)
Model.save('warmup_weights.keras')

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 131s 94ms/step - accuracy: 0.6776 - loss: 1.0071 - val_accuracy: 0.8863 - val_loss: 0.4063
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.7714 - loss: 0.7039 - val_accuracy: 0.9026 - val_loss: 0.3236
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 110s 88ms/step - accuracy: 0.8138 - loss: 0.5774 - val_accuracy: 0.9130 - val_loss: 0.2862
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 110s 88ms/step - accuracy: 0.8412 - loss: 0.4950 - val_accuracy: 0.9173 - val_loss: 0.2769
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 107s 85ms/step - accuracy: 0.8580 - loss: 0.4349 - val_accuracy: 0.9226 - val_loss: 0.2590


In [29]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

In [33]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0.0005,
    patience=5,
    restore_best_weights=True  # reverts to the best checkpoint
)
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=5e-5,
    decay_steps=2000,
    decay_rate=0.95
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = Model.fit(train_set, validation_data=valid_set, epochs=30, callbacks = [early_stop])

Epoch 1/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 132s 95ms/step - accuracy: 0.8876 - loss: 0.3471 - val_accuracy: 0.9274 - val_loss: 0.2463
Epoch 2/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.8848 - loss: 0.3463 - val_accuracy: 0.9278 - val_loss: 0.2463
Epoch 3/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.8887 - loss: 0.3453 - val_accuracy: 0.9283 - val_loss: 0.2442
Epoch 4/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.8890 - loss: 0.3404 - val_accuracy: 0.9281 - val_loss: 0.2444
Epoch 5/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.8897 - loss: 0.3357 - val_accuracy: 0.9277 - val_loss: 0.2449
Epoch 6/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 106s 85ms/step - accuracy: 0.8907 - loss: 0.3340 - val_accuracy: 0.9282 - val_loss: 0.2469
Epoch 7/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 109s 87ms/step - accuracy: 0.8903 - loss: 0.3341 - val_accuracy: 0.9281 - val_loss: 0.2474
Epoch 8/30
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 110s 88ms/step - accuracy: 

In [ ]:
# tf.keras.models.save_model(Model, 'my_model.keras')
# print("Model saved to my_model.keras")